# Практическая работа. Введение
## Данные и выборки: готовим данные для машинного обучения на чистом Python

**Цель:** своими руками написать все, что в следующих практических работах будет стоять в «готовой ячейке» в самом начале: генераторы данных беспилотника, перемешивание, разбиение на обучающую и тестовую выборки, подсчет классов и простейшие модели-ориентиры. После этой работы в тех ячейках не останется ни одной непонятной строки.

> **Правило работы.** Только встроенный Python: списки, словари, циклы, `zip`, `sum`, `min`, `max`, `len`, `sorted`, `round`. Единственный разрешенный модуль – `random`, и только через свой генератор `rnd = random.Random(seed)`. Никаких `numpy`, `pandas`, `sklearn` и готовых `random.shuffle`, `random.sample` внутри своих функций. Библиотеки появятся только в необязательной ячейке для сверки.

**Как работать с ноутбуком.** В ячейках с заданиями замените `...` своим кодом – подсказка написана в комментарии `# <--`. После функций идут строки проверки: напечатанное значение должно совпасть с тем, что стоит после слова «ожидается». Готовые ячейки (без `...`) просто выполните. Сначала прочитайте теорию ниже – все задания опираются на нее.

## Теория

### 1. Что такое машинное обучение

**Машинное обучение** (Machine Learning, ML) – это способ научить компьютер решать задачу не по готовой инструкции, а на опыте – на примерах данных. Система учится распознавать объекты, делать прогнозы и относить объекты к классам, хотя ни один шаг решения программист не прописывал.

Обычная программа – это **правила**, которые написал человек: «если скорость больше 60 км/ч и длина больше 3 м – это автомобиль». Для многих задач такие правила придумать невозможно: попробуйте словами описать, чем пешеход на видео отличается от велосипедиста в любую погоду и под любым углом. Машинное обучение переворачивает схему: человек дает компьютеру **примеры с правильными ответами**, а компьютер сам находит правило.

$$\text{программирование:}\quad \text{правила} + \text{данные} \to \text{ответы}, \qquad \text{машинное обучение:}\quad \text{данные} + \text{ответы} \to \text{правила}.$$

*Аналогия.* Двухлетнего ребенка отправили за килограммом картошки. Сначала его надо научить отличать картошку от других овощей, понимать, что такое килограмм, соотносить количество и цену. Даже с хорошими объяснениями с первого раза вряд ли получится – нужен опыт, много попыток. Алгоритму машинного обучения тоже нужно много примеров.

Найденное правило называется **моделью**. Модель ценна только тогда, когда правильно отвечает на **новых** объектах, которых она не видела. Это свойство называется **обобщающей способностью**, и вокруг него построена вся эта работа.

*Немного истории.* В 1958 году Фрэнк Розенблатт построил перцептрон – первую искусственную нейросеть, распознававшую образы. В 1959 году Артур Сэмюэл из IBM ввел сам термин «машинное обучение»: его программа для игры в шашки играла сама с собой и училась на своих партиях. В 1996–1997 годах компьютер Deep Blue обыграл чемпиона мира по шахматам Гарри Каспарова.

**Машинное обучение, ИИ и глубокое обучение.** Искусственный интеллект – общее направление: все технологии, которые решают «интеллектуальные» задачи. Машинное обучение – его часть, где правила не задаются заранее, а выводятся из данных. Глубокое обучение – часть машинного обучения, основанная на многослойных нейросетях: классическим методам признаки готовит человек, а глубокие сети умеют извлекать их сами из больших массивов данных (это тема 10 класса).

### 2. Объекты, признаки, ответы

**Объект** – то, про что мы хотим что-то узнать: один кадр с камеры, одна машина на дороге, один пациент. Объект описывается **признаками** – числами, которые мы про него знаем. Для объекта на дороге это, например, скорость, длина, высота и расстояние до него. **Ответ** (целевая переменная, метка) – то, что мы хотим предсказать: пешеход это, велосипедист или автомобиль.

Если объектов $N$, а признаков $n$, данные образуют таблицу «объект – признак»: $N$ строк и $n$ столбцов. В Python мы храним ее так:

- `X` – **список объектов**, каждый объект – список из $n$ чисел: `X[i][j]` – признак $j$ объекта $i$;
- `y` – **список ответов** длины $N$: `y[i]` – ответ для объекта `X[i]`.

| $i$ | скорость, км/ч | длина, м | высота, м | расстояние, м | ответ $y$ |
|---|---|---|---|---|---|
| 0 | 22,2 | 2,07 | 1,50 | 56,8 | 1 (велосипедист) |
| 1 | 5,7 | 0,77 | 1,86 | 64,4 | 0 (пешеход) |
| 2 | 15,9 | 4,51 | 0,97 | 41,7 | 2 (автомобиль) |

Главное правило хранения: **пара `X[i]`, `y[i]` неразрывна**. Если мы переставляем объекты, то ответы переставляем точно так же, иначе ответы «уедут» от своих объектов, и модель будет учиться на мусоре.

### 3. Виды обучения и виды задач

- **Обучение с учителем** – данные **размечены**: для каждого примера известен правильный ответ, и модель учится связывать признаки с ответами. Чтобы научить беспилотник находить машины, берут тысячи кадров, на которых люди обвели каждую машину рамкой. Задачи:
  - **классификация** – ответ из конечного набора классов: пешеход / велосипедист / автомобиль. Классы кодируем числами 0, 1, 2;
  - **регрессия** – ответ – число: тормозной путь в метрах, время до столкновения в секундах.
- **Обучение без учителя** – ответов нет, модель сама ищет закономерности. Главная задача – **кластеризация**, разбиение объектов на группы похожих: интернет-магазин замечает, что одни покупатели каждый месяц берут подгузники и игрушки, и относит их к группе «родители». Разберем в Блоке 2.
- **Обучение с подкреплением** – модель действует и получает за действия награду или штраф, а затем учится действовать так, чтобы награда была больше. Так учатся игровые программы и роботы: робот-манипулятор пробует открыть дверь и постепенно учится открывать ручки разной формы.

### 4. Как устроена работа над задачей машинного обучения

Работа специалиста по машинному обучению идет по пяти этапам. Модель – только четвертый из них, а большая часть времени уходит на данные.

1. **Сбор данных.** Решаем, какие данные нужны и что про объект важно знать. В этой работе данные мы генерируем сами (задания 1 и 3).
2. **Разметка.** Для обучения с учителем каждому примеру нужен правильный ответ. Люди обводят объекты рамками, подписывают классы, эксперты проверяют спорные случаи. Это долго и дорого, и полностью автоматизировать разметку нельзя.
3. **Исследовательский анализ.** Смотрим на данные: как распределены признаки, есть ли ошибки и странные значения, какие признаки связаны с ответом (задание 4).
4. **Обучение модели.** Выбираем несколько подходящих алгоритмов и обучаем их – этим займемся в следующих работах. Перед обучением данные делят на выборки (задания 5 и 6).
5. **Тестирование и оценка.** Сравниваем модели на отложенных данных и с ориентиром (задания 7 и 8), а затем решаем: собрать еще данные, поменять настройки или взять другой алгоритм.

### 5. Случайность и зерно (seed)

Компьютер не умеет бросать кубик: «случайные» числа он вычисляет по формуле из начального числа – **зерна** (seed). Одинаковое зерно дает одинаковую последовательность чисел. Это важно: эксперимент можно повторить, а у всех учеников получатся одни и те же числа.

Мы создаем собственный генератор `rnd = random.Random(seed)` и пользуемся тремя его методами:

- `rnd.uniform(a, b)` – случайное дробное число от $a$ до $b$, все значения одинаково вероятны;
- `rnd.gauss(mu, sigma)` – число из **нормального распределения**: чаще всего около среднего $\mu$, а примерно 95 % значений лежат в пределах $\mu \pm 2\sigma$. Так устроены многие реальные величины: рост людей, скорость пешеходов, ошибки измерений. Число $\sigma$ называют **разбросом** (стандартным отклонением);
- `rnd.randrange(m)` – случайное целое число от 0 до $m - 1$.

Важно: генератор помнит, сколько чисел уже выдал. Если поменять порядок вызовов, числа достанутся другим величинам, и данные получатся другими.

### 6. Перемешивание: алгоритм Фишера – Йейтса

Как честно перемешать список из $n$ элементов, чтобы все $n!$ порядков были равновероятны? Идем с конца:

1. для $i$ от $n - 1$ до $1$:
2. выбираем случайное $j$ от $0$ до $i$ включительно;
3. меняем местами элементы $i$ и $j$.

*Пример.* Список `[0, 1, 2, 3]`. Шаг $i = 3$: выпало $j = 1$, меняем, получаем `[0, 3, 2, 1]`. Шаг $i = 2$: выпало $j = 2$ – элемент остается на месте. Шаг $i = 1$: выпало $j = 0$ – получаем `[3, 0, 2, 1]`. Каждый элемент на каждом шаге может попасть на место $i$ с равной вероятностью – поэтому перемешивание честное. Именно этот алгоритм спрятан внутри `random.shuffle`.

Перемешиваем мы не сами объекты, а **номера** объектов (индексы) – так пары `X[i]`, `y[i]` гарантированно не разорвутся: по одному и тому же списку номеров собираем и `X`, и `y`.

### 7. Обучающая, валидационная и тестовая выборки

Как узнать, хорошо ли модель будет работать на новых объектах, если новых объектов у нас нет? Отложим часть данных и **не будем показывать ее модели** до самого конца:

| выборка | для чего | аналогия |
|---|---|---|
| **обучающая** (train) | по ней модель учится | задачи из учебника |
| **валидационная** (validation) | по ней выбираем настройки модели | пробный экзамен |
| **тестовая** (test) | один раз в конце оцениваем качество | настоящий экзамен |

Обычная доля теста – 20–30 %. Если объектов $N$, а доля теста `test_size`, то в тест идет $N_{test} = \lfloor N \cdot \text{test\_size} \rfloor$ объектов (целая часть, `int(...)`), остальные – в обучение. Для 300 объектов и `test_size = 0.3`: 90 в тест, 210 в обучение.

Четыре правила честной проверки:

1. **Перемешивать перед разбиением.** Данные часто записаны по порядку: сначала все пешеходы, потом все машины. Если отрезать последние 30 % без перемешивания, в тест попадут одни машины.
2. **Сохранять доли классов.** Если какого-то класса мало, при случайном разбиении в тест его может попасть слишком мало или ни одного. **Стратифицированное** разбиение делит каждый класс отдельно, поэтому доли классов в обучении и тесте совпадают.
3. **Не допускать утечки данных.** Утечка – когда информация из теста попадает в обучение: одинаковые объекты оказались по обе стороны разбиения, минимум и максимум для нормализации посчитаны по всем данным, настройки подобраны по тесту. Оценка качества после утечки слишком хорошая и лживая.
4. **Смотреть в тест один раз.** Все подборы – на валидации. Тест – экзамен, его не пересдают.

### 8. Модели-ориентиры и меры качества

Прежде чем радоваться качеству модели, нужно знать, с чем сравнивать. **Модель-ориентир** (baseline) – самая глупая модель, которая все равно что-то угадывает:

- для классификации – всегда отвечать **самым частым классом** обучающей выборки;
- для регрессии – всегда отвечать **средним** значением ответов обучающей выборки.

Любая настоящая модель обязана быть лучше ориентира, иначе она бесполезна. Качество меряем так:

$$\mathrm{accuracy} = \frac{\text{число верных ответов}}{N}, \qquad \mathrm{MSE} = \frac{1}{N}\sum_{i=1}^{N}(y_i - \hat y_i)^2, \qquad \mathrm{RMSE} = \sqrt{\mathrm{MSE}}.$$

Здесь $y_i$ – правильный ответ, $\hat y_i$ – прогноз. RMSE измеряется в тех же единицах, что и ответ (метры, секунды), поэтому его легко объяснить словами.

### 9. Описательная статистика

Первое, что делают с новыми данными, – смотрят на каждый признак: наименьшее и наибольшее значение, среднее и разброс.

$$\bar x = \frac{1}{N}\sum_{i=1}^{N} x_i, \qquad \sigma = \sqrt{\frac{1}{N}\sum_{i=1}^{N}(x_i - \bar x)^2}.$$

*Пример.* Значения $[2, 4, 4, 6]$: среднее $\bar x = 16 / 4 = 4$, отклонения $-2, 0, 0, 2$, их квадраты $4, 0, 0, 4$, $\sigma = \sqrt{8 / 4} = \sqrt 2 \approx 1{,}414$.

Если среднее признака почти одинаково у всех классов, признак, скорее всего, бесполезен для их различения.

In [ ]:
# Готовая ячейка: единственный разрешенный модуль и общее зерно для всей работы
import random

RANDOM_STATE = 42
CLASS_NAMES = ['пешеход', 'велосипедист', 'автомобиль']

# Маленькая учебная таблица для проверок: 6 объектов, 2 признака, 2 класса
X_toy = [[1, 10], [2, 20], [3, 30], [4, 40], [5, 50], [6, 60]]
y_toy = [0, 0, 0, 1, 1, 1]

# Как работает зерно: два генератора с одним зерном выдают одно и то же
r1, r2 = random.Random(7), random.Random(7)
print([round(r1.uniform(0, 1), 3) for _ in range(3)])
print([round(r2.uniform(0, 1), 3) for _ in range(3)], '  ожидается та же тройка чисел')

**Задание 1. Первый генератор данных: тормозной путь.**

Беспилотнику нужно знать тормозной путь автомобиля. Физика дает формулу: путь за время реакции $t_r$ плюс путь торможения

$$s = \frac{v}{3{,}6}\, t_r + \frac{v^2}{254\,\mu},$$

где $v$ – скорость в км/ч (деление на 3,6 переводит км/ч в м/с), $t_r = 1$ с – время реакции, $\mu = 0{,}7$ – сцепление шин с сухим асфальтом, $s$ – путь в метрах. Настоящие замеры никогда не ложатся точно на формулу: мешают ветер, уклон, износ шин. Поэтому к формуле добавим **шум** – случайную ошибку $\varepsilon$ из нормального распределения со средним 0 и разбросом $3 + 0{,}05\,v$ (на большой скорости ошибка больше).

Функция `make_braking(n, seed)` делает $n$ замеров. Для каждого замера **строго в таком порядке**:

1. скорость `v = rnd.uniform(10, 120)`;
2. путь по формуле `s = v / 3.6 * 1.0 + v ** 2 / (254 * 0.7)`;
3. в `X` добавляем объект из одного признака `[round(v, 1)]`, в `y` – ответ `round(s + rnd.gauss(0, 3 + 0.05 * v), 1)`.

Округление до десятых – как у настоящего датчика.

*Посчитайте на бумаге до запуска:* при $v = 60$ км/ч путь без шума $60 / 3{,}6 + 3600 / 177{,}8 \approx 16{,}7 + 20{,}2 = 36{,}9$ м.

In [ ]:
def make_braking(n=80, seed=RANDOM_STATE):
    """Скорость автомобиля (км/ч) -> тормозной путь (м) с учетом реакции водителя 1 с."""
    rnd = random.Random(seed)
    X, y = [], []
    for _ in range(n):
        v = ...                                   # <-- случайная скорость от 10 до 120
        s = ...                                   # <-- путь по формуле
        X.append(...)                             # <-- объект из одного признака
        y.append(...)                             # <-- путь с шумом, до десятых
    return X, y


X_b, y_b = make_braking()
print(len(X_b), len(y_b), '  ожидается 80 80')
print(X_b[:3], '  ожидается [[80.3], [34.6], [91.0]]')
print(y_b[:3], '  ожидается [64.2, 16.9, 64.8]')
print(make_braking(5, seed=1) == make_braking(5, seed=1), '  ожидается True (то же зерно – те же данные)')
print(make_braking(5, seed=1) == make_braking(5, seed=2), '  ожидается False')

**Вопрос 1.** а) Что будет, если в цикле сначала вызвать `rnd.gauss`, а потом `rnd.uniform`? Изменятся ли данные, хотя формулы те же? б) Зачем добавлять к формуле шум, если формула известна? в) Модель, которую мы обучим на этих данных, будет знать формулу? Что она будет знать на самом деле?

*Ваш ответ:*

**Задание 2. Перемешивание своими руками.**

Функция `shuffle_indices(n, rnd)` возвращает номера `0, 1, …, n − 1` в случайном порядке по алгоритму Фишера – Йейтса (п. 6 теории):

- создаем список номеров `idx = list(range(n))`;
- идем по $i$ от $n - 1$ вниз до 1: `range(n - 1, 0, -1)`;
- выбираем $j$ от 0 до $i$ включительно: `rnd.randrange(i + 1)`;
- меняем местами: в Python это одна строка `idx[i], idx[j] = idx[j], idx[i]`.

Генератор `rnd` передается в функцию снаружи: так одна и та же последовательность случайных чисел может использоваться в нескольких функциях подряд.

Проверка покажет, что наш алгоритм выдает **ровно тот же** порядок, что и `random.shuffle` при том же зерне, – значит, внутри библиотеки спрятан именно он.

In [ ]:
def shuffle_indices(n, rnd):
    idx = list(range(n))
    for i in range(n - 1, 0, -1):
        j = ...                               # <-- случайное целое от 0 до i включительно
        idx[i], idx[j] = ...                  # <-- поменяйте местами idx[i] и idx[j]
    return idx


p = shuffle_indices(10, random.Random(RANDOM_STATE))
print(p, '  ожидается [7, 3, 2, 8, 5, 6, 9, 4, 0, 1]')
print(sorted(p) == list(range(10)), '  ожидается True (каждый номер ровно один раз)')

library = list(range(300))
random.Random(RANDOM_STATE).shuffle(library)
print(shuffle_indices(300, random.Random(RANDOM_STATE)) == library, '  ожидается True (совпадает с random.shuffle)')

In [ ]:
# Готовая ячейка: честно ли перемешивание? Где оказывается элемент 0 в 6000 перемешиваниях списка из 3 элементов
rnd_test = random.Random(1)
where = [0, 0, 0]
for _ in range(6000):
    where[shuffle_indices(3, rnd_test).index(0)] += 1
print(where, '  ожидается примерно [2000, 2000, 2000]')

**Вопрос 2.** а) Почему перемешиваем номера объектов, а не сами списки `X` и `y` по отдельности? б) Что сломается, если выбирать $j$ из всего списка (`rnd.randrange(n)`), а не от 0 до $i$? (Подсказка: для $n = 3$ такой алгоритм делает $3^3 = 27$ равновероятных исходов, а порядков всего $3! = 6$. Делится ли 27 на 6?)

*Ваш ответ:*

**Задание 3. Генератор данных беспилотника.**

Лидар и камеры беспилотника выделяют на дороге объекты и измеряют у каждого четыре признака: скорость (км/ч), длину (м), высоту (м) и расстояние до него (м). Классы: 0 – пешеход, 1 – велосипедист, 2 – автомобиль. Каждый признак класса задан парой (среднее, разброс) – словарь `params` дан готовым:

| класс | скорость | длина | высота |
|---|---|---|---|
| пешеход | 5 ± 2,5 | 0,7 ± 0,4 | 1,7 ± 0,15 |
| велосипедист | 14 ± 6 | 1,7 ± 0,45 | 1,75 ± 0,15 |
| автомобиль | 18 ± 12 | 3,6 ± 1,1 | 1,6 ± 0,2 |

Расстояние до объекта от класса **не зависит**: `rnd.uniform(5, 80)` для всех. Это сделано нарочно – посмотрим, заметим ли мы бесполезный признак.

Для каждого класса `c` и каждой пары `(pv, pl, ph)` делаем `n_per_class` объектов, **строго в таком порядке**:

1. `v = max(0.5, rnd.gauss(*pv))` – скорость; `max` не дает ей стать меньше 0,5 (нормальное распределение иногда дает отрицательные числа, а отрицательная скорость – бессмыслица). Запись `rnd.gauss(*pv)` раскрывает пару: то же, что `rnd.gauss(pv[0], pv[1])`;
2. `L = max(0.2, rnd.gauss(*pl))` – длина;
3. `H = max(0.8, rnd.gauss(*ph))` – высота;
4. `D = rnd.uniform(5, 80)` – расстояние;
5. объект `[round(v, 1), round(L, 2), round(H, 2), round(D, 1)]` в `X`, класс `c` в `y`.

После цикла объекты идут по порядку: 100 пешеходов, 100 велосипедистов, 100 машин. Перемешайте их своей функцией `shuffle_indices` **тем же генератором** `rnd` и соберите `X` и `y` по перемешанным номерам.

In [ ]:
def make_road_objects(n_per_class=100, seed=RANDOM_STATE):
    """Объекты, которые «видит» беспилотник: 0 – пешеход, 1 – велосипедист, 2 – автомобиль.
    Признаки: [скорость, км/ч;  длина, м;  высота, м;  расстояние до объекта, м]."""
    rnd = random.Random(seed)
    params = {0: ((5, 2.5), (0.7, 0.4), (1.7, 0.15)),     # (среднее, разброс) скорости, длины, высоты
              1: ((14, 6), (1.7, 0.45), (1.75, 0.15)),
              2: ((18, 12), (3.6, 1.1), (1.6, 0.2))}
    X, y = [], []
    for c, (pv, pl, ph) in params.items():
        for _ in range(n_per_class):
            v = ...                            # <-- скорость, не меньше 0.5
            L = ...                            # <-- длина, не меньше 0.2
            H = ...                            # <-- высота, не меньше 0.8
            D = ...                            # <-- расстояние от 5 до 80, от класса не зависит
            X.append([round(v, 1), round(L, 2), round(H, 2), round(D, 1)])
            y.append(...)                      # <-- метка класса
    idx = ...                                  # <-- перемешанные номера, тем же генератором rnd
    return ...                                    # <-- два списка по номерам idx: [X[i] for i in idx] и так же для y


X_road, y_road = make_road_objects()
print(len(X_road), len(y_road), '  ожидается 300 300')
print(X_road[0], CLASS_NAMES[y_road[0]], '  ожидается [22.2, 2.07, 1.5, 56.8] велосипедист')
print(y_road[:10], '  ожидается [1, 0, 2, 0, 2, 1, 0, 0, 2, 2]')

**Вопрос 3.** а) Зачем ограничивать скорость снизу через `max(0.5, ...)`? Какие еще признаки нуждаются в таком ограничении и почему? б) Почему перемешивать нужно тем же генератором `rnd`, а не новым `random.Random(seed)`? в) Настоящие данные беспилотника никто не генерирует – их размечают люди. Какие ошибки могут быть в настоящей разметке?

*Ваш ответ:*

**Задание 4. Разведка данных.**

Прежде чем учить модель, на данные нужно посмотреть. Напишите четыре функции:

- `column(X, j)` – список значений признака $j$ у всех объектов: `[row[j] for row in X]`;
- `describe(values)` – словарь `{'min': ..., 'max': ..., 'mean': ..., 'std': ...}` по формулам п. 9 теории; корень – это `** 0.5`;
- `class_counts(y)` – словарь «класс → число объектов», как в голосовании kNN: `counts.get(c, 0) + 1`;
- `mean_by_class(X, y, j)` – словарь «класс → среднее признака $j$ у объектов этого класса». Удобно собрать для каждого класса список значений, а потом взять средние.

*Проверка на бумаге:* для $[2, 4, 4, 6]$ ответ `{'min': 2, 'max': 6, 'mean': 4.0, 'std': 1.414}`.

In [ ]:
def column(X, j):
    return ...                                # <-- значения признака j у всех объектов

def describe(values):
    n = len(values)
    mean = ...                                # <-- среднее
    var = ...                                     # <-- средний квадрат отклонения
    return {'min': min(values), 'max': max(values), 'mean': mean, 'std': var ** 0.5}

def class_counts(y):
    counts = {}
    for c in y:
        counts[c] = ...                       # <-- добавьте один объект классу c
    return counts

def mean_by_class(X, y, j):
    groups = {}
    for row, c in zip(X, y):
        groups.setdefault(c, []).append(row[j])    # список значений признака j для класса c
    return ...                                    # <-- словарь {c: среднее vals} для c, vals в sorted(groups.items())


print(column(X_toy, 1), '  ожидается [10, 20, 30, 40, 50, 60]')
d = describe([2, 4, 4, 6])
print({k: round(v, 3) for k, v in d.items()}, "  ожидается {'min': 2, 'max': 6, 'mean': 4.0, 'std': 1.414}")
print(class_counts(y_road), '  ожидается по 100 объектов каждого класса')
print({c: round(m, 2) for c, m in mean_by_class(X_toy, y_toy, 0).items()}, '  ожидается {0: 2.0, 1: 5.0}')

In [ ]:
# Готовая ячейка: сводка по признакам данных беспилотника
FEATURES = ['скорость, км/ч', 'длина, м', 'высота, м', 'расстояние, м']
print(f"{'признак':16}{'min':>8}{'max':>8}{'mean':>8}{'std':>8}   {'среднее по классам: пешеход / велосипедист / автомобиль'}")
for j, name in enumerate(FEATURES):
    d = describe(column(X_road, j))
    m = mean_by_class(X_road, y_road, j)
    print(f"{name:16}{d['min']:8.2f}{d['max']:8.2f}{d['mean']:8.2f}{d['std']:8.2f}   "
          + ' / '.join(f'{m[c]:.2f}' for c in range(3)))
print('ожидается: средняя длина 0.69 / 1.75 / 3.60, среднее расстояние 43.69 / 44.00 / 44.06')

**Вопрос 4.** а) Какой признак лучше всего отличает автомобиль от пешехода? Какой признак почти бесполезен и как это видно из таблицы? б) Скорость меняется от 0,5 до 39,2, а высота – от 1 до 2,2. Чем это может помешать методу, который складывает разности признаков? в) Почему у автомобилей разброс скорости такой большой?

*Ваш ответ:*

**Задание 5. Обучающая и тестовая выборки.**

Функция `train_test_split(X, y, test_size, seed)` делит данные случайно (п. 7 теории):

1. перемешать номера своей `shuffle_indices(len(X), random.Random(seed))`;
2. число тестовых объектов `n_test = int(len(X) * test_size)`;
3. первые `n_test` номеров – тест, остальные – обучение (срезы `idx[:n_test]` и `idx[n_test:]`);
4. вернуть **четыре** списка в порядке `X_train, X_test, y_train, y_test`.

Функция `split_no_shuffle(X, y, test_size)` – «плохое» разбиение без перемешивания: последние `n_test` объектов – тест, остальные – обучение. Она нужна, чтобы увидеть, почему так делать нельзя.

После проверки убедимся, что ваше разбиение **совпадает** с готовой ячейкой из работы по kNN: там первый обучающий объект – `[37.1, 4.52, 1.78, 77.9]`, автомобиль.

In [ ]:
def train_test_split(X, y, test_size=0.3, seed=RANDOM_STATE):
    """Перемешивает объекты и делит их на обучающую и тестовую части."""
    idx = ...                                     # <-- перемешанные номера
    n_test = ...                                  # <-- сколько объектов в тест
    test, train = ...                             # <-- срезы idx[:n_test] и idx[n_test:]
    return ([X[i] for i in train], [X[i] for i in test],
            [y[i] for i in train], [y[i] for i in test])

def split_no_shuffle(X, y, test_size=0.3):
    n_test = int(len(X) * test_size)
    n_train = len(X) - n_test
    return ...                                    # <-- четыре среза: X[:n_train], X[n_train:], и так же для y


X_train, X_test, y_train, y_test = train_test_split(X_road, y_road)
print(len(X_train), len(X_test), '  ожидается 210 90')
print(X_train[0], CLASS_NAMES[y_train[0]], '  ожидается [37.1, 4.52, 1.78, 77.9] автомобиль')
print(class_counts(y_train), class_counts(y_test), '  ожидается {2: 74, 0: 67, 1: 69} {0: 33, 1: 31, 2: 26}')

# Отложим от обучающей выборки валидационную – как при подборе k в работе по kNN
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.3, seed=13)
print(len(X_tr), len(X_val), len(X_test), '  ожидается 147 63 90')

# Плохое разбиение: данные упорядочены по классам, как до перемешивания в генераторе
X_sorted = [x for x, c in sorted(zip(X_road, y_road), key=lambda p: p[1])]
y_sorted = sorted(y_road)
_, _, yb_train, yb_test = split_no_shuffle(X_sorted, y_sorted)
print(class_counts(yb_train), class_counts(yb_test), '  ожидается {0: 100, 1: 100, 2: 10} {2: 90}')

In [ ]:
# Готовая ячейка (необязательная): сверка с sklearn, если он установлен.
# sklearn перемешивает по-своему, поэтому объекты другие, но размеры и смысл те же.
try:
    from sklearn.model_selection import train_test_split as sk_split
    a, b, c, d = sk_split(X_road, y_road, test_size=0.3, random_state=RANDOM_STATE)
    print(len(a), len(b), '  ожидается 210 90')
except ImportError:
    print('sklearn не установлен – сверку можно пропустить')

**Вопрос 5.** а) Модель, обученная на «плохом» разбиении, видела всего 10 автомобилей, а в тесте одни автомобили. Что покажет такая проверка? б) В каком порядке обычно записываются реальные данные беспилотника (подсказка: заезд за заездом, день за днем)? Чем это опасно? в) Почему `int(len(X) * test_size)`, а не просто `len(X) * test_size`?

*Ваш ответ:*

**Задание 6. Стратифицированное разбиение.**

Когда класс редкий, случайное разбиение может положить его в тест слишком мало. Смоделируем это: возьмем всех пешеходов и автомобили, но только 10 велосипедистов – велосипедисты составят меньше 5 % данных.

Функция `stratified_split(X, y, test_size, seed)` делит **каждый класс отдельно** (п. 7 теории):

1. создаем один генератор `rnd = random.Random(seed)`;
2. для каждого класса `c` по порядку (`sorted(class_counts(y))`) собираем номера его объектов `idx_c = [i for i in range(len(y)) if y[i] == c]`;
3. перемешиваем их: `perm = shuffle_indices(len(idx_c), rnd)` дает порядок, в котором брать номера из `idx_c`;
4. первые `int(len(idx_c) * test_size)` номеров класса – в тест, остальные – в обучение;
5. в конце классы идут подряд, поэтому перемешиваем оба списка номеров еще раз тем же `rnd` и собираем выборки.

Готовая ячейка после функции сравнит, сколько велосипедистов попадает в тест при обычном и при стратифицированном разбиении для 10 разных зерен.

In [ ]:
def stratified_split(X, y, test_size=0.3, seed=RANDOM_STATE):
    rnd = random.Random(seed)
    train, test = [], []
    for c in sorted(class_counts(y)):
        idx_c = ...                               # <-- [i for i in range(len(y)) if y[i] == c]
        perm = shuffle_indices(len(idx_c), rnd)
        idx_c = [idx_c[p] for p in perm]                     # номера класса c в случайном порядке
        n_test_c = ...                            # <-- сколько объектов класса c в тест
        test += ...                               # <-- первые n_test_c – в тест
        train += ...                              # <-- остальные – в обучение
    train = [train[p] for p in shuffle_indices(len(train), rnd)]
    test = [test[p] for p in shuffle_indices(len(test), rnd)]
    return ([X[i] for i in train], [X[i] for i in test],
            [y[i] for i in train], [y[i] for i in test])


a, b, c, d = stratified_split(X_toy, y_toy, test_size=0.34, seed=0)
print(len(a), len(b), class_counts(d), '  ожидается 4 2 {0: 1, 1: 1} (по одному объекту каждого класса)')

# Редкий класс: все пешеходы и автомобили, но только 10 велосипедистов
keep = [i for i in range(len(y_road)) if y_road[i] != 1] + [i for i in range(len(y_road)) if y_road[i] == 1][:10]
X_rare = [X_road[i] for i in keep]
y_rare = [y_road[i] for i in keep]
print(len(y_rare), class_counts(y_rare), '  ожидается 210 {0: 100, 2: 100, 1: 10}')

In [ ]:
# Готовая ячейка: сколько велосипедистов (класс 1) попадает в тест из 63 объектов
plain = [class_counts(train_test_split(X_rare, y_rare, 0.3, seed=s)[3]).get(1, 0) for s in range(10)]
strat = [class_counts(stratified_split(X_rare, y_rare, 0.3, seed=s)[3]).get(1, 0) for s in range(10)]
print('обычное разбиение:        ', plain, '  ожидается [1, 3, 4, 3, 6, 2, 2, 1, 4, 2]')
print('стратифицированное:       ', strat, '  ожидается [3, 3, 3, 3, 3, 3, 3, 3, 3, 3]')

**Вопрос 6.** а) При обычном разбиении в тест попадало от 1 до 6 велосипедистов. Почему это плохо для оценки модели? Что если бы не попало ни одного? б) Почему при стратифицированном разбиении всегда ровно 3? в) Беспилотник должен замечать редкие, но опасные объекты: ребенка, животное, упавший груз. Почему для таких данных стратификация особенно важна?

*Ваш ответ:*

**Задание 7. Модель-ориентир и модель-зубрила.**

Напишите:

- `most_frequent(y)` – самый частый класс (при равенстве – меньший номер класса). Используйте свою `class_counts`: из словаря выберите ключ с наибольшим значением;
- `accuracy(y_true, y_pred)` – доля верных ответов (п. 8 теории);
- `memorizer_fit(X, y)` – «обучение» зубрилы: словарь, где ключ – объект, а значение – его ответ. Список нельзя сделать ключом словаря, поэтому объект превращаем в кортеж: `tuple(x)`;
- `memorizer_predict(model, X, default)` – для каждого объекта: если он есть в словаре, вернуть запомненный ответ, иначе – ответ `default`. Метод `model.get(tuple(x), default)` делает это одной строкой.

Зубрила идеально выучил обучающую выборку. Посмотрим, как он сдаст экзамен.

In [ ]:
def most_frequent(y):
    counts = class_counts(y)
    best = ...                                    # <-- наибольшее число объектов
    return ...                                    # <-- min из классов c, у которых counts[c] == best

def accuracy(y_true, y_pred):
    correct = sum(1 for a, b in zip(y_true, y_pred) if a == b)
    return ...                                    # <-- доля верных ответов

def memorizer_fit(X, y):
    model = {}
    for x, c in zip(X, y):
        model[tuple(x)] = ...                     # <-- запомнить ответ объекта
    return model

def memorizer_predict(model, X, default):
    return ...                                    # <-- список: model.get(tuple(x), default) для каждого x из X


print(most_frequent([2, 0, 2, 1, 0]), '  ожидается 0 (у классов 0 и 2 поровну, берем меньший)')
print(accuracy([0, 1, 2, 1], [0, 1, 1, 1]), '  ожидается 0.75')

base = most_frequent(y_train)
acc_base = accuracy(y_test, [base] * len(y_test))
print('ориентир: всегда', CLASS_NAMES[base], '– accuracy на тесте', round(acc_base, 3), '  ожидается автомобиль, 0.289')

mem = memorizer_fit(X_train, y_train)
acc_mem_train = accuracy(y_train, memorizer_predict(mem, X_train, base))
acc_mem_test = accuracy(y_test, memorizer_predict(mem, X_test, base))
print('зубрила: обучение', acc_mem_train, ' тест', round(acc_mem_test, 3), '  ожидается 1.0 и 0.289')

Зубрила получил 100 % на обучении и ровно столько же, сколько ориентир, на тесте: новых объектов он никогда не видел и на каждый отвечает «автомобиль». Выучить ответы – не значит научиться. Теперь посмотрим, как **утечка данных** делает зубрилу «гением».

Представим, что при сборе данных каждый объект случайно записали дважды (например, два датчика прислали один и тот же кадр). Разобьем такие данные и посчитаем, сколько тестовых объектов есть и в обучающей выборке.

In [ ]:
def count_leaks(X_train, X_test):
    seen = set(tuple(x) for x in X_train)
    return ...                                    # <-- сколько x из X_test таких, что tuple(x) in seen


print(count_leaks(X_train, X_test), '  ожидается 0 (честное разбиение)')

X_dup, y_dup = X_road + X_road, y_road + y_road               # каждый объект записан дважды
Xd_train, Xd_test, yd_train, yd_test = train_test_split(X_dup, y_dup)
print(len(Xd_test), count_leaks(Xd_train, Xd_test), '  ожидается 180 и 134 (утечка!)')
mem_d = memorizer_fit(Xd_train, yd_train)
acc_leak = accuracy(yd_test, memorizer_predict(mem_d, Xd_test, most_frequent(yd_train)))
print('зубрила на данных с утечкой: тест', round(acc_leak, 3), '  ожидается 0.811')

**Вопрос 7.** а) Зачем вообще считать точность ориентира, если он заведомо глупый? б) Почему зубрила получил на тесте ровно столько же, сколько ориентир? в) На данных с утечкой зубрила «сдал экзамен» на 0,811. Чему на самом деле научилась модель? Как обнаружить утечку до обучения?

*Ваш ответ:*

**Задание 8. Ориентир для регрессии.**

Для тормозного пути ориентир – всегда отвечать **средним** путем обучающей выборки. Напишите:

- `mean(values)` – среднее значение;
- `mse(y_true, y_pred)` – среднеквадратичная ошибка (п. 8 теории).

Затем разобьем замеры тормозного пути и посчитаем RMSE ориентира на тесте. Это число – планка: в работе по kNN модель получает RMSE около 6,5 м, и теперь будет ясно, с чем его сравнивать.

*Посчитайте на бумаге:* `mse([1, 2, 3], [1, 4, 0])` – ошибки $0, -2, 3$, квадраты $0, 4, 9$, среднее $13 / 3 \approx 4{,}333$.

In [ ]:
def mean(values):
    return ...                                # <-- среднее

def mse(y_true, y_pred):
    s = 0
    for a, b in zip(y_true, y_pred):
        s += ...                              # <-- квадрат ошибки
    return ...                                # <-- среднее


print(mean([2, 4, 4, 6]), '  ожидается 4.0')
print(round(mse([1, 2, 3], [1, 4, 0]), 3), '  ожидается 4.333')

Xb_train, Xb_test, yb_train, yb_test = train_test_split(X_b, y_b)
m = mean(yb_train)
rmse_base = mse(yb_test, [m] * len(yb_test)) ** 0.5
print(len(Xb_train), len(Xb_test), '  ожидается 56 24')
print('ориентир: всегда', round(m, 1), 'м, RMSE на тесте', round(rmse_base, 1), 'м', '  ожидается 45.1 м и 34.0 м')

In [ ]:
# Готовая ячейка (необязательная): как выглядят данные и ориентир
try:
    import matplotlib.pyplot as plt
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
    for c, col in zip(range(3), ['#9A9AA3', '#1C1C22', '#B8AB84']):
        pts = [x for x, t in zip(X_road, y_road) if t == c]
        axs[0].scatter(column(pts, 0), column(pts, 1), s=12, color=col, label=CLASS_NAMES[c])
    axs[0].set_xlabel('скорость, км/ч'); axs[0].set_ylabel('длина, м'); axs[0].legend(); axs[0].set_title('Объекты на дороге')
    axs[1].scatter(column(Xb_train, 0), yb_train, s=14, color='#1C1C22', label='обучение')
    axs[1].scatter(column(Xb_test, 0), yb_test, s=14, color='#B8AB84', label='тест')
    axs[1].axhline(m, color='#8F8260', linestyle='--', label='ориентир: среднее')
    axs[1].set_xlabel('скорость, км/ч'); axs[1].set_ylabel('тормозной путь, м'); axs[1].legend(); axs[1].set_title('Тормозной путь')
    plt.tight_layout(); plt.show()
except ImportError:
    print('matplotlib не установлен – график можно пропустить')

**Вопрос 8.** а) RMSE ориентира – 34 м. Объясните это число словами: насколько в среднем ошибается модель «всегда 45,1 м»? б) Где ориентир ошибается сильнее всего – на малых или на больших скоростях? в) Что должна использовать настоящая модель, чтобы обогнать ориентир?

*Ваш ответ:*

### Итог: готовая ячейка следующих работ

Теперь вы можете прочитать и написать с нуля готовую ячейку, с которой начинаются следующие практические работы. Все ее части – ваши функции:

| строка готовой ячейки | ваше задание |
|---|---|
| `random.Random(seed)`, `rnd.gauss`, `rnd.uniform` | задание 1, п. 5 теории |
| `random.Random(seed).shuffle(idx)` | задание 2 – `shuffle_indices` |
| `make_road_objects`, `make_braking` | задания 3 и 1 |
| `train_test_split` | задание 5 |
| `print(len(X_train), len(X_test))` – ожидается 210 90 | задание 5 |

Последняя проверка: ваши данные совпадают с данными следующих работ.

In [ ]:
print(len(X_train), len(X_test), '  ожидается 210 90')
print(X_train[0], CLASS_NAMES[y_train[0]], '  ожидается [37.1, 4.52, 1.78, 77.9] автомобиль')

### Задание повышенной сложности

1. **Кросс-валидация.** Когда данных мало, одной валидационной выборки мало: результат зависит от того, какие объекты в нее попали. Напишите `kfold_indices(n, k, seed)`, которая перемешивает номера и делит их на $k$ примерно равных блоков; возвращает список из $k$ пар `(train_idx, val_idx)`, где каждый блок по очереди – валидация. Проверьте, что каждый номер ровно один раз попадает в валидацию. Для $n = 10$, $k = 3$ размеры блоков – 4, 3, 3.
2. **Коварная арифметика.** Посчитайте `int(100 * 0.29)`. Почему получилось 28, а не 29? Предложите, как исправить `n_test`, чтобы такого не было.
3. **Устойчивость оценки.** Посчитайте точность ориентира на тесте для 20 разных зерен разбиения данных беспилотника. Какой разброс у этой величины? Что это значит для сравнения двух моделей, которые отличаются по точности на 0,02?
4. **Утечка при нормализации.** В работе по kNN признаки переводят в отрезок $[0, 1]$ по формуле $(x - \min) / (\max - \min)$. Объясните, почему $\min$ и $\max$ нужно считать только по обучающей выборке. Придумайте пример, когда расчет по всем данным дает модели подсказку о тесте.

In [ ]:
# Место для задания повышенной сложности